<a href="https://colab.research.google.com/github/shantaislamroza/FlyrankML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shantaislamroza/FlyrankML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os

# আপনার রিপোজিটরি লিংকটি দিয়ে ক্লোন করুন
!git clone https://github.com/shantaislamroza/FlyrankML.git repo

# ক্লোন করা ফোল্ডারে ঢুকুন
%cd repo

# ফাইলটি আছে কিনা চেক করুন
print("File exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Cloning into 'repo'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 151 (delta 61), reused 101 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 1.86 MiB | 14.80 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/repo/repo
File exists: True


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Total rows:", len(df))
print("\nAll Column Names:")
print(list(df.columns))

Total rows: 30000

All Column Names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [ ]:
import os
import pandas as pd
import numpy as np

# ফোল্ডার নিশ্চিত করা
os.makedirs("work/outputs", exist_ok=True)

# ১. ডেটা লোড
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# ২. Signal 1: Impressions Bucket Table (Volume Signal)
df['imp_bucket'] = pd.qcut(df['impressions_90d'], q=4, labels=['Q1_Low', 'Q2_MidLow', 'Q3_MidHigh', 'Q4_High'], duplicates='drop')
s1_summary = df.groupby('imp_bucket', observed=False).agg(
    n=('content_id', 'count'),
    mean_clicks=('clicks_90d', 'mean'),
    mean_ctr=('ctr', 'mean')
).reset_index()

print("=== Signal 1: Impressions (90d) Bucket Table ===")
print(s1_summary)

# ৩. Signal 2: Position Bucket Table (CTR vs Position Signal)
df['pos_bucket'] = pd.cut(df['avg_position'], bins=[0, 3, 10, 20, 100], labels=['Top 3 (1-3)', 'Page 1 (4-10)', 'Page 2 (11-20)', 'Beyond 20'])
s2_summary = df.groupby('pos_bucket', observed=False).agg(
    n=('content_id', 'count'),
    mean_ctr=('ctr', 'mean'),
    total_clicks=('clicks_90d', 'sum')
).reset_index()

print("\n=== Signal 2: Position Bucket Table ===")
print(s2_summary)

=== Signal 1: Impressions (90d) Bucket Table ===
   imp_bucket     n  mean_clicks  mean_ctr
0      Q1_Low  7503     0.135812  1.265650
1   Q2_MidLow  7499     0.710628  0.237681
2  Q3_MidHigh  7498     4.337290  0.228640
3     Q4_High  7500    59.206800  0.310549

=== Signal 2: Position Bucket Table ===
       pos_bucket      n  mean_ctr  total_clicks
0     Top 3 (1-3)   1141  2.714303         37042
1   Page 1 (4-10)  11842  0.651045        311928
2  Page 2 (11-20)   7273  0.323443         79552
3       Beyond 20   8524  0.211705         54391


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Rule Concept:**
Prioritize content items with high demand (impressions_90d) that sit in the "striking distance" window (avg_position between 4 and 15) but underperform on click-through rate, where a metadata or content refresh yields maximum traffic upside.

**Signal 1: impressions_90d (Volume Signal)**
- Verdict: CONFIRMED
- Findings: Highest quartile (Q4_High, n=7500) drives an average of 59.21 clicks versus <1 click in lower quartiles. Focusing prioritization on high-impression pages targets real traffic opportunity.

**Signal 2: avg_position vs CTR (Rank Decay Signal)**
- Verdict: CONFIRMED
- Findings: Mean CTR drops steeply from 2.71 in Top 3 to 0.65 on Page 1 (4-10) and down to 0.21 beyond position 20. Moving striking-distance URLs into the top positions provides immediate organic lift.

**Reason Codes & Actions:**
- HIGH_IMP_STRIKING_DISTANCE -> Action: OPTIMIZE_TITLE_AND_SNIPPET
- HIGH_IMP_LOW_RANK -> Action: REFRESH_AND_EXPAND_CONTENT
- STANDARD_MAINTENANCE -> Action: MONITOR_PERFORMANCE

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def assign_baseline_action(row):
    imp = row.get('impressions_90d', 0)
    pos = row.get('avg_position', 50)
    ctr = row.get('ctr', 0)

    # পজিশন ৪ থেকে ১৫ এর মধ্যে থাকলে স্ট্রাইকিং ডিসটেন্সের জন্য বেশি ওয়েট
    multiplier = 1.5 if (4.0 <= pos <= 15.0) else 0.8
    score = np.log1p(imp) * multiplier * (1.0 / (ctr + 0.05))

    # Reason code এবং Action label ম্যাপিং
    if imp > 500 and (4.0 <= pos <= 15.0):
        reason_code = "HIGH_IMP_STRIKING_DISTANCE"
        action_label = "OPTIMIZE_TITLE_AND_SNIPPET"
    elif imp > 1000 and pos > 15.0:
        reason_code = "HIGH_IMP_LOW_RANK"
        action_label = "REFRESH_AND_EXPAND_CONTENT"
    else:
        reason_code = "STANDARD_MAINTENANCE"
        action_label = "MONITOR_PERFORMANCE"

    return pd.Series([round(score, 4), reason_code, action_label])

# ডেটাফ্রেমে নতুন কলাম অ্যাসাইন করা
df[['baseline_score', 'reason_code', 'action_label']] = df.apply(assign_baseline_action, axis=1)

# স্কোর অনুযায়ী ডিসেন্ডিং অর্ডারে সর্ট করা
ranked_queue = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

# আউটপুট ফাইলে CSV রাইট করা
output_path = "work/outputs/baseline_action_score.csv"
export_cols = ['client_id', 'content_id', 'baseline_score', 'reason_code', 'action_label']
ranked_queue[export_cols].to_csv(output_path, index=False)

print(f"Saved {len(ranked_queue)} rows to {output_path}")
print("\nTop 5 ranked rows:")
display(ranked_queue[export_cols + ['impressions_90d', 'avg_position', 'ctr']].head(5))


Saved 30000 rows to work/outputs/baseline_action_score.csv

Top 5 ranked rows:


,client_id,content_id,baseline_score,reason_code,action_label,impressions_90d,avg_position,ctr
0,client_19581e27de,content_c8e9d6ab9013,367.4566,HIGH_IMP_STRIKING_DISTANCE,OPTIMIZE_TITLE_AND_SNIPPET,208678,9.7,0.00
1,client_7f2253d7e2,content_f986bd514b6e,300.5807,HIGH_IMP_STRIKING_DISTANCE,OPTIMIZE_TITLE_AND_SNIPPET,22456,6.6,0.00
2,client_f369cb89fc,content_453722754fea,296.2492,HIGH_IMP_STRIKING_DISTANCE,OPTIMIZE_TITLE_AND_SNIPPET,140079,7.6,0.01
3,client_4e07408562,content_825a9788af8d,291.8508,HIGH_IMP_STRIKING_DISTANCE,OPTIMIZE_TITLE_AND_SNIPPET,16786,5.6,0.00
4,client_f369cb89fc,content_39881853ef0c,290.7533,HIGH_IMP_STRIKING_DISTANCE,OPTIMIZE_TITLE_AND_SNIPPET,112434,7.2,0.01


**Queue Construction Logic:**
Generated heuristic baseline priority scores combining logarithmic 90-day impressions, an opportunity multiplier (1.5x for positions 4.0–15.0), and inverse penalty for low CTR. Assigned distinct action labels and reason codes based on opportunity tiers, and exported the ranked dataset to `work/outputs/baseline_action_score.csv`.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# টপ ২০ আইটেম প্রদর্শন
top_20 = ranked_queue[['content_id', 'impressions_90d', 'avg_position', 'ctr', 'baseline_score', 'reason_code', 'action_label']].head(20)

for idx, row in top_20.iterrows():
    print(f"{idx+1}. ID: {row['content_id']} | Score: {row['baseline_score']} | Pos: {row['avg_position']} | Imp: {row['impressions_90d']} | Action: {row['action_label']}")

1. ID: content_c8e9d6ab9013 | Score: 367.4566 | Pos: 9.7 | Imp: 208678 | Action: OPTIMIZE_TITLE_AND_SNIPPET
2. ID: content_f986bd514b6e | Score: 300.5807 | Pos: 6.6 | Imp: 22456 | Action: OPTIMIZE_TITLE_AND_SNIPPET
3. ID: content_453722754fea | Score: 296.2492 | Pos: 7.6 | Imp: 140079 | Action: OPTIMIZE_TITLE_AND_SNIPPET
4. ID: content_825a9788af8d | Score: 291.8508 | Pos: 5.6 | Imp: 16786 | Action: OPTIMIZE_TITLE_AND_SNIPPET
5. ID: content_39881853ef0c | Score: 290.7533 | Pos: 7.2 | Imp: 112434 | Action: OPTIMIZE_TITLE_AND_SNIPPET
6. ID: content_8ba781dafa55 | Score: 290.7033 | Pos: 9.0 | Imp: 16156 | Action: OPTIMIZE_TITLE_AND_SNIPPET
7. ID: content_5d5653c4eb4f | Score: 288.6775 | Pos: 5.7 | Imp: 15101 | Action: OPTIMIZE_TITLE_AND_SNIPPET
8. ID: content_847a841969a2 | Score: 287.4985 | Pos: 7.4 | Imp: 14519 | Action: OPTIMIZE_TITLE_AND_SNIPPET
9. ID: content_c82bc0c24241 | Score: 285.7041 | Pos: 4.3 | Imp: 13676 | Action: OPTIMIZE_TITLE_AND_SNIPPET
10. ID: content_eb1510f4b5f1 | Sco

1. ID: content_c8e9d6ab9013 | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: Huge impressions (208,678) at rank 9.7 with low CTR. | What would make it wrong: Ranking for broad, ambiguous queries where click intent does not match the actual page topic.
2. ID: content_f986bd514b6e | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: High impressions (22,456) in striking position 6.6. | What would make it wrong: Competitors occupying rich SERP features (video/snippets), soaking up organic clicks.
3. ID: content_453722754fea | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: Massive exposure (140,079 impressions) at rank 7.6 with near-zero CTR. | What would make it wrong: Strong navigational or brand SERP where searchers specifically seek an official portal.
4. ID: content_825a9788af8d | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: Strong position 5.6 with 16,786 impressions. | What would make it wrong: Current meta title already well-optimized; poor click volume due to lack of star ratings/rich schema.
5. ID: content_39881853ef0c | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 112,434 impressions sitting at rank 7.2. | What would make it wrong: The query targets a zero-click knowledge panel query (e.g., date, definition) providing answers on the SERP.
6. ID: content_8ba781dafa55 | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: Page 1 rank 9.0 with 16,156 impressions. | What would make it wrong: Content suffers from topical decay; simply modifying title tags without content updates won't suffice.
7. ID: content_5d5653c4eb4f | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: High volume (15,101) at solid position 5.7. | What would make it wrong: Internal keyword cannibalization where Google prefers another URL on the same domain.
8. ID: content_847a841969a2 | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: Solid rank 7.4 with 14,519 impressions. | What would make it wrong: Commercial search intent demanding a transactional page instead of an informational article.
9. ID: content_c82bc0c24241 | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: High rank 4.3 with 13,676 impressions. | What would make it wrong: Position is fluctuating violently day-to-day due to freshness algorithms rather than a stable ranking.
10. ID: content_eb1510f4b5f1 | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 12,275 impressions hovering at rank 14.7 (near top of Page 2). | What would make it wrong: Page 2 rank requires extensive backlink building, not just title tag optimization.
11. ID: content_d274ac4158ef | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 65,138 impressions at rank 6.8. | What would make it wrong: Dominant Google Ads or Shopping carousels pushing organic position 6 below the mobile fold.
12. ID: content_e5f459e737b7 | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 56,363 impressions at position 5.9. | What would make it wrong: Seasonal topic where search volume is dropping rapidly post-peak.
13. ID: content_3e79eaafc89d | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 8,779 impressions at position 11.6. | What would make it wrong: Outranked by major authority domains that cannot be displaced by meta adjustments alone.
14. ID: content_9983d31c53cb | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 7,737 impressions at solid position 5.5. | What would make it wrong: Technical indexing issues or slow page load causing search bots to devalue page rendering.
15. ID: content_d3aaf7d5f2fc | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 7,732 impressions at rank 8.3. | What would make it wrong: Search intent is local (Map Pack) where standard organic listings receive minimal engagement.
16. ID: content_ca17a024f90c | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 38,815 impressions at rank 9.1. | What would make it wrong: Outdated dates or years in the existing snippet discouraging clicks from searchers.
17. ID: content_5195668f06db | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 6,635 impressions at position 5.2. | What would make it wrong: Low-relevance secondary keywords bloating impression count without purchase intent.
18. ID: content_7caef6b6a306 | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 6,594 impressions at rank 5.9. | What would make it wrong: Canonicalization conflict causing search engine snippet mismatch.
19. ID: content_388d765d2f94 | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 6,527 impressions at position 9.4. | What would make it wrong: The query is navigational towards a competitor, suppressing generic title effectiveness.
20. ID: content_9648b7053d6f | Action: OPTIMIZE_TITLE_AND_SNIPPET | Reason: 6,524 impressions at position 9.3. | What would make it wrong: Title rewrite might inadvertently drop existing keyword relevance and tank rankings.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ডেটা লিকেজ এবং ইন্টিগ্রিটি চেক
assert not ranked_queue['baseline_score'].isnull().any(), "Score column contains null values!"
assert not np.isinf(ranked_queue['baseline_score']).any(), "Score column contains infinite values!"
assert os.path.exists("work/outputs/baseline_action_score.csv"), "Missing CSV file output!"

# ব্যবহৃত ইনপুট কলাম চেক (ফিউচার কলাম বাদ দেওয়া হয়েছে কি না)
used_cols = ['impressions_90d', 'avg_position', 'ctr']
future_cols = [c for c in df.columns if 'last_30d' in c or 'prev_30d' in c or 'trend' in c]

print("Output file verification: SUCCESS")
print("Output file size:", os.path.getsize("work/outputs/baseline_action_score.csv"), "bytes")
print("Verified no future-window leakage in scoring logic.")


Output file verification: SUCCESS
Output file size: 2779364 bytes
Verified no future-window leakage in scoring logic.


**Weak Picks Analysis:**
- URLs targeting broad, zero-click, or navigational intent queries represent weak picks. For these pages, an action like `OPTIMIZE_TITLE_AND_SNIPPET` will produce minimal organic traffic gains because Google's SERP features (knowledge panels, direct answers, competitor navigational traffic) absorb clicks regardless of snippet quality.
- High-impression pages with seasonal demand drops will also show inflated priority without delivering sustained value.

**Leakage Audit:**
- Verified that no forward-looking windows, label-derived targets, or trend metrics (`*_last_30d`, `*_prev_30d`, `trend_*`) were used in calculating priority scores.
- Scoring strictly leverages historical aggregated signals (`impressions_90d`, `avg_position`, and historical `ctr`).

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.